## Section 1: What Is a Decorator? 

In [1]:
#A decorator is a function that takes another function as input, adds some extra behaviour to it, and returns a new (enhanced) function — WITHOUT changing the original function's code.
#One-line Definition decorator = a wrapper function that modifies or extends another function's behaviour transparently. 

Section 2: Function Decorators — Built from Scratch 

In [ ]:
# step 1:Functions are objects
# In Python, functions are first-class objects. You can store them in variables and pass them to other functions. 

# Functions are stored in a variable
def greet():
    print("Hello")
say_hello=greet    # say_hello now points to the same function 
say_hello()         #Output: Hello!

Hello


In [4]:
# Step 2: A function inside a function(closure)
# A function can be defined inside another function. The inner function can access variables from the outer function's scope — this is called a closure. 
 
def outer():
    message="I am the outer function"
    def inner():
        print(message)
    inner()
outer()

I am the outer function


In [5]:
# Step 3 Returning a function
# A function can return another function as its result. This is the heart of how decorators work. 

# Returning a function
def make_greeter():
    def say_hi():
        print("Hi there!")
    return say_hi
my_func=make_greeter()
my_func()


Hi there!


In [7]:
# Step 4:Building a real decorator
# Now combine everything: a function that takes a function, defines a wrapper inside, and returns the wrapper.

def dec(func):
    def wrap():
        print("Before the function runs")
        func()
        print("After the function runs")
    return wrap
def say_hello():
    print("Hello World")
say_hello=dec(say_hello)
say_hello()

Before the function runs
Hello World
After the function runs


Decorators with *args and **kwargs 

In [8]:
# Decorators with *args and **kwargs 
# The wrapper above only works for functions with no arguments. To make a decorator work for ANY function, always use *args and **kwargs in the wrapper. 

def dec(func):
    def wrap(*args,**kwargs):
        print("before")
        result=func(*args,**kwargs)
        print("After")
        return result
    return wrap
def add(a,b):
    return a+b
add=dec(add)
print(add(3,8))

before
After
11


# Section 3: The @decorator Syntax 

In [14]:
# Python provides a cleaner, more readable shorthand for applying decorators: the @ symbol placed on the line immediately above the function definition. 

def shout(func):
    def wrap(*args,**kwargs):
        result=func(*args,**kwargs)
        return str(result).upper()
    return wrap
@shout
def greet(name):
    return f"hello, {name}"
print(greet("Abhi"))

HELLO, ABHI


# Stacking Multiple Decorators 

In [15]:
# Stacking Multiple Decorators 
# You can stack multiple decorators on a single function. They are applied from BOTTOM to TOP (innermost first). 
def dec1(func):
    def wrap1(*args,**kwargs):
        func(*args,**kwargs)
    return wrap1
def dec2(func):
    def wrap2(*args,**kwargs):
        func(*args,**kwargs)
    return wrap2
@dec1
@dec2
def message():
    print("Hello")
message()

Hello


# functools.wraps — Preserving Identity 

In [16]:
# functools.wraps — Preserving Identity 
# By default, a decorator replaces the original function with wrapper, losing the original function's name and docstring. The @functools.wraps fix preserves them — always use it in production code. 
 
# without functools
def dec(func):
    def wrap(*args,**kwargs):
        func(*args,**kwargs)
    return wrap
@dec
def greet():
    print("Abhi")
print(greet.__name__)

wrap


In [ ]:
# With functools wraps(correct way)
import functools
def dec(func):
    @functools.wraps(func)
    def wrap(*args,**kwargs):
        func(*args,**kwargs)
    return wrap
@dec
def greet():
    print("Hi")
print(greet.__name__)

greet


## Section 4: Decorators with Arguments

In [ ]:
# Section 4: Decorators with Arguments
#  Sometimes you want to pass configuration arguments to the decorator itself — not to the function being decorated. This requires one extra level of nesting: a decorator factory. 

#  Structure: Three Levels of Nesting 
def dec1(args1):
    def dec(func):
        def wrap1(*args,**kwargs):
            func(*args,**kwargs)
        return wrap1
    return dec
@dec1(say_hello)


Concrete Example — Repeat Decorator 

In [20]:
# Concrete Example — Repeat Decorator 
#A decorator that runs a function n times:
def dec(times):
    def dec1(func):
        @functools.wraps(func)
        def wrap1(*args,**kwargs):
            for i in range(times):
                result=func(*args,**kwargs)
            return result
        return wrap1
    return dec1
@dec(3)
def say_hello():
    print("Hello!")
say_hello()

Hello!
Hello!
Hello!


In [ ]:
def dec(times):
    def dec1(func):
        @functools.wraps(func)
        def wrap1(*args,**kwargs):
            for i in range(times):
                func(*args,**kwargs)
        return wrap1
    return dec1
def say_hello():
    print("Hello!")
say_hello=dec(3)(say_hello)
say_hello()

Hello!
Hello!
Hello!


In [2]:
# Decorator with Required + Optional Arguments 

import functools

def prefix_log(prefix,separator='|'):
    def dec(func):
        @functools.wraps(func)
        def wrap(*args,**kwargs):
            result=func(*args,**kwargs)
            print(f"{prefix} {separator} {result}")
            # return result
        return wrap
    return dec
@prefix_log('INFO')
def get_status():
    return 'Server is running'
@prefix_log('ERROR',separator='-->')
def get_error():
    return '404 not found'
get_status()
get_error()

INFO | Server is running
ERROR --> 404 not found


### Section 5: Real-World Use Cases 

In [6]:
# 5.1  Logging Decorator 
# A logging decorator automatically records when a function is called, what arguments it received, and what it returned — without touching the function's code.
import functools
import datetime
def logger(func):
    @functools.wraps(func)
    def wrap(*args,**kwargs):
        timestamp=datetime.datetime.now().strftime('%H:%M:%S')
        print(f'[{timestamp}] calling:{func.__name__}')
        print(f'Args:{args},Kwargs:{kwargs}')
        result=func(*args,**kwargs)
        print(f' Returned:{result}')
        # return result
    return wrap
@logger
def multiply(a,b):
    return a*b
@logger
def greet(name,greeting='Hello'):
    return f'{greeting},{name}!'
multiply(4,7)
print()
greet("Abhi",greeting='Hi')

[17:22:49] calling:multiply
Args:(4, 7),Kwargs:{}
 Returned:28

[17:22:49] calling:greet
Args:('Abhi',),Kwargs:{'greeting': 'Hi'}
 Returned:Hi,Abhi!


In [11]:
import datetime
time=datetime.datetime.now()
print("Year:",time.year)
print("Month:",time.month)
print("Day:",time.day)
print("hour:",time.hour)
print("minute:",time.minute)
print("seconds:",time.second)
date_time=datetime.datetime.now().strftime("%H:%M:%S %d:%m:%Y")
print(date_time)


Year: 2026
Month: 8
Day: 18
hour: 17
minute: 35
seconds: 35
17:35:35 18:08:2026


In [2]:
# 5.2  Authentication Decorator 
# Authentication decorators check whether a user has the right to run a function before allowing execution. This pattern is used in every web framework. 
import functools
def require_role(required_role):
    def dec(func):
        @functools.wraps(func)
        def wrap(user,*args,**kwargs):
            if(user.get('role')!= required_role):
                print(f"Access Denied: {user["name"]} is not a {required_role}")
                return None
            print(f"Access Granted:{user["name"]}")
            return func(user,*args,**kwargs)
        return wrap
    return dec
@require_role('admin')
def delete_database(user):
    print(f"{user["name"]} deleted the database.")
@require_role('student')
def view_notes(user):
    print(f"{user["name"]} is viewing notes.")
admin={'name':'alice','role':'admin'}
guest={'name':'bob','role':'guest'}
student={'name':'carol','role':'student'}

delete_database(admin)
delete_database(guest)
view_notes(student)                                

Access Granted:alice
alice deleted the database.
Access Denied: bob is not a admin
Access Granted:carol
carol is viewing notes.


In [3]:
# 5.3  Timing / Performance Decorator

In [ ]:
# Q1.  In your own words, what is a decorator? Write a one-paragraph explanation without looking at your notes.


#A decorator is a function that takes another function as input, adds some extra behaviour to it, and returns a new (enhanced) function — WITHOUT changing the original function's code.
#decorator = a wrapper function that modifies or extends another function's behaviour transparently.

In [4]:
# Q2.  Write a simple decorator called my_decorator that prints 'Function is starting' before and 'Function is done' after any function it wraps. Apply it to a function greet() that prints 'Hello!'. 
def my_decorator(func):
    def wrap():
        print("Function is starting")
        func()
        print("Function is done")
    return wrap
def greet():
    print("Hello!")
greet=my_decorator(greet)
greet()

Function is starting
Hello!
Function is done


In [5]:
# Q3.  What is the purpose of *args and **kwargs in the wrapper function inside a decorator? Why is it important to include them? 

#*args and **kwargs allow a decorator's wrapper function to accept any number of positional and keyword arguments. They are important because they make the decorator flexible and allow it to work with functions having different parameters.

In [ ]:
# Q4.  What does @functools.wraps(func) do? Write an example showing what happens to __name__ with and without it. 
# with @functools.wraps(func)
import functools
def dec(func):
    @functools.wraps(func)
    def wrap(*args,**kwargs):
        func(*args,**kwargs)
    return wrap
@dec
def greet():
    print("Hi")
print(greet.__name__)

# without @functools.wraps(func)
def dec1(func):
    def wrap1(*args,**kwargs):
        func(*args,**kwargs)
    return wrap1
@dec1
def great():
    print("Hello")
print(great.__name__)

greet
wrap1


In [14]:
#Q1.  Write a decorator called validate_positive that checks all positional arguments passed to a function. If any argument is negative, print an error message and return None without calling the function. Test it on a function multiply(a, b). 
def validate_positive(func):
    def wrap(*args,**kwargs):
        for i in args:
            if(i<0):
                print("error message")
                return None
        return func(*args,**kwargs)
    return wrap
@validate_positive
def mul(a,b):
    return a*b
print(mul(5,4))
print(mul(-6,7))

20
error message
None


In [15]:
# Q2.  Create a decorator factory called repeat(n) that runs the decorated function exactly n times. Then stack it with a logger decorator and apply both to a function. Trace the exact order in which the wrappers execute. 
def repeat(n):
    def dec(func):
        def wrap1(*args,**kwargs):
            for i in range(n):
                func(*args,**kwargs)
        return wrap1
    return dec
def dec1(func):
    def wrap(*args,**kwargs):
        print("logger")
        func(*args,**kwargs)
    return wrap
@repeat(3)
@dec1
def greet():
    print("Hello")
greet()


logger
Hello
logger
Hello
logger
Hello


In [16]:
# Q3.  Write a decorator called count_calls that tracks how many times a function has been called and prints the count each time.
#  Hint: you will need to store state — use a mutable object (like a list or a dictionary attribute on the wrapper).
def count_calls(func):
    count=[0]
    def wrap(*args,**kwargs):
        count[0]+=1
        print("Function is called:",count[0])
        func(*args,**kwargs)
    return wrap
@count_calls
def greet():
    print("Hello")
greet()
greet()
greet()


Function is called: 1
Hello
Function is called: 2
Hello
Function is called: 3
Hello


In [23]:
# Q4.  Explain the difference between these two usages and what error (if any) the wrong one produces:     Option A:  @my_decorator     Option B:  @my_decorator()     When is each form correct? 
def my_decorator(func):
    def wrap(*args,**kwargs):
        func(*args,**kwargs)
    return wrap
@my_decorator
def greet():
    print("Hi")
greet()
# TypeError: my_decorator() missing 1 required positional argument: 'func'
# @my_decorator is used when the decorator directly accepts the function.
#  @my_decorator() is used when the decorator is a decorator factory that must be called first, usually to pass arguments. Using the wrong form can result in a TypeError.

Hi
